# Yoke — Learning Path

This notebook is the entry point for the 30 walkthrough notebooks under `examples/sNN_*/`. Each example isolates **one capability** of the harness; the order is from simplest (a bare model loop) to most complex (real-world Kubernetes integration).

Read this notebook first, pick a tier or one of the curated paths at the bottom, then open the linked notebooks in order. Cells inside each leaf notebook are designed to be run top-to-bottom.

## Before you start

All notebooks share the same Jupyter kernel and the same environment expectations.

**Kernel** — [GoNB](https://github.com/janpfeifer/gonb):
```bash
go install github.com/janpfeifer/gonb@latest
gonb --install
```

**Provider** — every notebook calls a real LLM:
```bash
export YOKE_PROVIDER=openai_compat
export OPENAI_BASE_URL=http://localhost:11434/v1
export YOKE_MODEL=qwen2.5-coder
```
(defaults to a local Ollama; swap in `anthropic` / `openai` / `gemini` for hosted providers — see [docs/providers.md](../docs/providers.md))

**Run jupyter from the repo root** so module imports resolve:
```bash
cd /path/to/yoke && jupyter lab
```

More setup detail in [docs/notebooks.md](../docs/notebooks.md).

## How each notebook is laid out

Every leaf notebook follows the same skeleton, so once you've read one you can skim the rest:

1. **Title + concept anchor** — what this example teaches, where it fits in the harness.
2. **Prerequisites** — env vars and optional extras (Redis for [s29_redis](s29_redis/s29_redis.ipynb), a cluster for [s30_k8s_context_e2e](s30_k8s_context_e2e/s30_k8s_context_e2e.ipynb), etc.).
3. **Imports + helper** cells.
4. **Numbered walkthrough** (~3-5 sections) — each chunk of the underlying `main.go` split into a markdown intro + code cell pair.
5. **What to look for** — observable behaviour to confirm.
6. **Try it yourself** — one or two short variations.

---

## Tier 1 — Single-tool basics

**Theme:** the smallest possible agent surface. By the end of this tier you understand the loop, you've seen the model call a single tool, and you've worked with the file/shell toolkit. Everything later is just *more participants in the same loop*.

| Notebook | What you'll learn |
|---|---|
| [s01_loop](s01_loop/s01_loop.ipynb) | The bare `model → tool → model` cycle. No tools, so the loop terminates after one turn. **Start here.** |
| [s02_calc](s02_calc/s02_calc.ipynb) | A single, deterministic tool (`calculate`). Models are bad at arithmetic — offload it. |
| [s03_mime](s03_mime/s03_mime.ipynb) | A single inspection tool (`mime`). Magic-byte detection vs. filename extension. |
| [s04_stream](s04_stream/s04_stream.ipynb) | Token-by-token streaming via `RunOnceStream`. Same loop, different output transport. |
| [s05_tools](s05_tools/s05_tools.ipynb) | The full file/shell toolkit: `bash`, `read`, `write`, `grep`, `glob`, `revert`. The everyday workbench. |
| [s06_revert](s06_revert/s06_revert.ipynb) | Zoom in on `revert` — how the write tool snapshots so the agent can undo itself. |
| [s07_web_search](s07_web_search/s07_web_search.ipynb) | Reaching outside the local fs: `web_search` (DDG/SerpAPI) + `web_fetch`. |

## Tier 2 — Tool-ecosystem extensions

**Theme:** the toolkit is open-ended. You can plug in interactivity, post-process tool output, or load entire third-party tool catalogs.

| Notebook | What you'll learn |
|---|---|
| [s08_ask_user](s08_ask_user/s08_ask_user.ipynb) | The agent *pauses the loop* and asks the human (single/multi/text/confirm). The askuser registry is the indirection that makes web/TUI/console all work the same way. |
| [s09_output_filters](s09_output_filters/s09_output_filters.ipynb) | YAML rules (`config/filters/*.yaml`) that condense noisy bash output **before** it hits the model. See the byte-count delta live. |
| [s10_mcp](s10_mcp/s10_mcp.ipynb) | Mount external [Model Context Protocol](https://modelcontextprotocol.io) servers as ADK tools. Specialisation without writing Go. |

## Tier 3 — Session state

**Theme:** the agent stops being purely reactive. It plans, remembers, runs work in the background, parallelises, recovers, and lets you abort.

| Notebook | What you'll learn |
|---|---|
| [s11_todo](s11_todo/s11_todo.ipynb) | A scratchpad: `todo_write` / `todo_read` / `todo_update`. Cheap, flat, ephemeral. |
| [s12_tasks](s12_tasks/s12_tasks.ipynb) | A durable task graph with `depends_on`. Survives across runs; JSON file under `logs/`. |
| [s13_bg](s13_bg/s13_bg.ipynb) | `bash_background` — fire-and-forget commands with out-of-band completion notifications. The agent stops waiting. |
| [s14_cache](s14_cache/s14_cache.ipynb) | Prompt-cache stats plugin. Hit rate is the cheapest performance metric you'll get. |
| [s15_parallel](s15_parallel/s15_parallel.ipynb) | Multiple tool calls dispatched in one model turn — ADK runs them in parallel. |
| [s16_resume](s16_resume/s16_resume.ipynb) | Re-use a session ID; the conversation is reloaded transparently. |
| [s17_interrupt](s17_interrupt/s17_interrupt.ipynb) | Ctrl-C / `context.Cancel` mid-run. No goroutine leaks, no half-completed tool calls. |

## Tier 4 — Cross-cutting plugins

**Theme:** governance and observability layered on top of the loop. Plugins watch every step without changing the agent's code.

| Notebook | What you'll learn |
|---|---|
| [s18_events](s18_events/s18_events.ipynb) | The event bus — `session_start`, before/after model/tool, `compress_*`, `curate_now`, etc. The single audit-log entry point. |
| [s19_permissions](s19_permissions/s19_permissions.ipynb) | YAML-driven gating: `deny → ask → allow` (Claude Code nomenclature). The human-in-the-loop's seatbelt. |
| [s20_compress](s20_compress/s20_compress.ipynb) | Soft (75%) and hard (92%) compression triggers, plus the task-switch heuristic. Keeps long sessions usable. |

## Tier 5 — Specialisation & multi-agent

**Theme:** stop being a generalist. Load domain knowledge (skills), delegate (sub-agents), or learn from past sessions (curator).

| Notebook | What you'll learn |
|---|---|
| [s21_skills](s21_skills/s21_skills.ipynb) | Lazy `SKILL.md` loading from `./skills/`. Human-authored playbooks. |
| [s22_subagents](s22_subagents/s22_subagents.ipynb) | Wrap an agent as a tool via `agenttool.New`. Control returns to the leader after every call. |
| [s23_softskills](s23_softskills/s23_softskills.ipynb) | The curator turns past sessions into new skills, automatically. Closes the loop on learning. |

## Tier 6 — Multi-process & distributed

**Theme:** isolation and coordination. The agent works in its own git worktree; multiple agents talk via a mailbox; the mailbox can be local or Redis-backed.

| Notebook | What you'll learn |
|---|---|
| [s24_worktree](s24_worktree/s24_worktree.ipynb) | `worktree_create` / `_remove` / `_merge` — the agent's sandbox. |
| [s25_conflicts](s25_conflicts/s25_conflicts.ipynb) | Programmatic merge-conflict scenario. The contract: `Merge` aborts cleanly with a conflict report. |
| [s26_mailbox](s26_mailbox/s26_mailbox.ipynb) | Persistent teammate mailbox (in-memory) — `teammate_ask` / `_tell` / `_inbox`. |
| [s27_fsm](s27_fsm/s27_fsm.ipynb) | The FSM protocol end-to-end: `IDLE → REQUESTING → WAITING → RESPONDING`. |
| [s28_self_assign](s28_self_assign/s28_self_assign.ipynb) | An autonomous worker goroutine pulls tasks from a shared queue. |
| [s29_redis](s29_redis/s29_redis.ipynb) | Same mailbox, Redis backend. Cross-process coordination, ~3 lines of change. |

## Tier 7 — End-to-end

**Theme:** what a real production scenario looks like when every component is wired together.

| Notebook | What you'll learn |
|---|---|
| [s30_k8s_context_e2e](s30_k8s_context_e2e/s30_k8s_context_e2e.ipynb) | Kubernetes context-compression validated against a real cluster, with pass/fail markers. The capstone integration test. |

---

## Curated paths

If the linear order is too much, pick the path that matches your goal:

### "I just want to run an agent against my files"
[s01_loop](s01_loop/s01_loop.ipynb) → [s05_tools](s05_tools/s05_tools.ipynb) → [s11_todo](s11_todo/s11_todo.ipynb) → [s18_events](s18_events/s18_events.ipynb)

### "I want to specialise the agent for my domain"
[s01_loop](s01_loop/s01_loop.ipynb) → [s05_tools](s05_tools/s05_tools.ipynb) → [s21_skills](s21_skills/s21_skills.ipynb) → [s10_mcp](s10_mcp/s10_mcp.ipynb) → [s19_permissions](s19_permissions/s19_permissions.ipynb) → [s22_subagents](s22_subagents/s22_subagents.ipynb)

### "I want to build a multi-agent system"
[s11_todo](s11_todo/s11_todo.ipynb) → [s12_tasks](s12_tasks/s12_tasks.ipynb) → [s13_bg](s13_bg/s13_bg.ipynb) → [s26_mailbox](s26_mailbox/s26_mailbox.ipynb) → [s27_fsm](s27_fsm/s27_fsm.ipynb) → [s28_self_assign](s28_self_assign/s28_self_assign.ipynb) → [s29_redis](s29_redis/s29_redis.ipynb)

### "I'm doing ops / SRE work"
[s05_tools](s05_tools/s05_tools.ipynb) → [s09_output_filters](s09_output_filters/s09_output_filters.ipynb) → [s13_bg](s13_bg/s13_bg.ipynb) → [s24_worktree](s24_worktree/s24_worktree.ipynb) → [s20_compress](s20_compress/s20_compress.ipynb) → [s30_k8s_context_e2e](s30_k8s_context_e2e/s30_k8s_context_e2e.ipynb)

### "I care about cost & latency"
[s14_cache](s14_cache/s14_cache.ipynb) → [s09_output_filters](s09_output_filters/s09_output_filters.ipynb) → [s15_parallel](s15_parallel/s15_parallel.ipynb) → [s20_compress](s20_compress/s20_compress.ipynb)

### "I want a self-improving agent"
[s18_events](s18_events/s18_events.ipynb) → [s20_compress](s20_compress/s20_compress.ipynb) → [s21_skills](s21_skills/s21_skills.ipynb) → [s23_softskills](s23_softskills/s23_softskills.ipynb)

---

## Verify the notebook set

Run the cell below to confirm all 30 leaf notebooks are present. Any missing file means the corresponding link above will 404.

In [ ]:
import (
    "fmt"
    "os"
    "path/filepath"
    "sort"
    "strings"
)

In [ ]:
entries, err := os.ReadDir(".")
if err != nil {
    panic(err)
}
var found []string
for _, e := range entries {
    if !e.IsDir() || !strings.HasPrefix(e.Name(), "s") {
        continue
    }
    nb := filepath.Join(e.Name(), e.Name()+".ipynb")
    status := "missing"
    if _, err := os.Stat(nb); err == nil {
        status = "ok"
    }
    found = append(found, fmt.Sprintf("%-30s %s", e.Name(), status))
}
sort.Strings(found)
for _, line := range found {
    fmt.Println(line)
}
fmt.Printf("\n%d notebooks discovered\n", len(found))

---

## Reading rhythm

- **One tier per session.** Three notebooks back-to-back is plenty before the concepts start blurring.
- **Don't skip "what to look for".** That section is where the *insight* lives; the code is just plumbing.
- **Try the variations.** Twisting one knob (different prompt, removed instruction, changed window size) teaches more than running the original 10 times.